# 🎵 ACE-Step v1.5 — Music Generation on Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-BF667%2Facestep--v1.5-blue)](https://github.com/BF667/acestep-v1.5) [![HuggingFace](https://img.shields.io/badge/%F0%9F%A4%97%20HuggingFace-ACE--Step%2FAce--Step1.5-yellow)](https://huggingface.co/ACE-Step/Ace-Step1.5)

Welcome to the **ACE-Step v1.5** Colab notebook! ACE-Step is a state-of-the-art music generation model that supports:

| Feature | Description |
|---------|-------------|
| **Text2Music** | Generate music from text descriptions & lyrics |
| **Cover** | Create covers of songs with modified style |
| **Repaint** | Re-generate specific sections of a song |
| **Remix** | Combine cover + repaint for style transfer |
| **Lego** | Generate individual instrument tracks |
| **Complete** | Continue/complete partial music |

### Requirements
- **GPU**: This notebook requires a **CUDA GPU** (free Colab T4 is supported)
- **Time**: First run downloads ~15 GB of models from HuggingFace Hub
- **VRAM**: CPU offload is enabled by default for T4 (16 GB VRAM)

### Quick Start
Run cells 1–4 in order to set up the environment and load models, then jump to any generation cell you like!

## Cell 1: Check GPU & Install System Dependencies

This cell verifies GPU availability and installs system-level dependencies like `ffmpeg`.

In [ ]:
#@title ## 1. Check GPU & Install System Dependencies

import subprocess
import sys

# ---- Check GPU ----
print("=" * 60)
print("GPU Check")
print("=" * 60)

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)

if gpu_info.returncode == 0:
    print(f"\u2705 GPU detected:\n  {gpu_info.stdout.strip()}")
else:
    print("\u274c No GPU found! Go to Runtime > Change runtime type > T4 GPU")
    sys.exit(1)

# ---- Install ffmpeg ----
print("\n" + "=" * 60)
print("Installing system dependencies...")
print("=" * 60)

!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("\u2705 ffmpeg installed")

# ---- Verify Python version ----
import platform
py_ver = platform.python_version()
print(f"\u2705 Python {py_ver}")
if not py_ver.startswith("3.1"):
    print(f"\u26a0\ufe0f  Recommended: Python 3.11.x (you have {py_ver})")

print("\n\u2705 System setup complete!")

## Cell 2: Clone the Repository

Clones the ACE-Step v1.5 repository from GitHub.

In [ ]:
#@title ## 2. Clone the Repository

import os

REPO_URL = "https://github.com/BF667/acestep-v1.5"
PROJECT_DIR = "/content/acestep-v1.5"

if os.path.exists(PROJECT_DIR):
    print(f"\u2705 Repository already exists at {PROJECT_DIR}")
    # Pull latest changes
    !cd {PROJECT_DIR} && git pull --ff-only 2>/dev/null || echo "\u26a0\ufe0f  Could not pull (local changes?)"
else:
    print(f"Cloning {REPO_URL} ...")
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
    print("\u2705 Repository cloned")

# Add nano-vllm to Python path
nano_vllm_path = os.path.join(PROJECT_DIR, "acestep", "third_parts", "nano-vllm")
if nano_vllm_path not in sys.path:
    sys.path.insert(0, nano_vllm_path)

# Add project root to path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"\nProject directory: {PROJECT_DIR}")
print("\u2705 Ready for dependency installation")

## Cell 3: Install Python Dependencies

Installs all required Python packages. We install PyTorch with CUDA 12.x support separately, then install the remaining dependencies from the project's `requirements.txt`.

In [ ]:
#@title ## 3. Install Python Dependencies

import subprocess

print("=" * 60)
print("Installing Python dependencies...")
print("=" * 60)

# ---- Step 1: Install PyTorch with CUDA support ----
print("\n[1/3] Installing PyTorch with CUDA 12.x support...")
!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu124

# Verify CUDA
import torch
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA device: {torch.cuda.get_device_name(0)}")

# ---- Step 2: Install core dependencies ----
print("\n[2/3] Installing core dependencies...")
!pip install -q \
    transformers>=4.51.0 \
    diffusers \
    gradio \
    matplotlib>=3.7.5 \
    scipy>=1.10.1 \
    soundfile>=0.13.1 \
    ffmpeg-python \
    loguru>=0.7.3 \
    einops>=0.8.1 \
    accelerate>=1.12.0 \
    diskcache \
    numba>=0.63.1 \
    vector-quantize-pytorch>=1.27.15 \
    huggingface_hub>=0.20.0 \
    pyyaml \
    xxhash

# ---- Step 3: Install triton (Linux only) ----
print("\n[3/3] Installing triton...")
!pip install -q triton>=3.0.0 2>/dev/null || echo "  Triton install skipped (non-critical)"

print("\n" + "=" * 60)
print("\u2705 All dependencies installed!")
print("=" * 60)

## Cell 4: Initialize Models

This cell initializes both the **DiT (Diffusion Transformer)** model and the **5Hz Language Model**. On first run, models are automatically downloaded from HuggingFace Hub (~15 GB). CPU offload is enabled for T4 GPUs to reduce VRAM usage.

### Available Models
| Model | Type | Description |
|-------|------|-------------|
| `acestep-v15-turbo` | DiT | **Default** — Fast inference, 8 steps |
| `acestep-v15-base` | DiT | Base model, 32–100 steps |
| `acestep-v15-sft` | DiT | SFT fine-tuned |
| `acestep-v15-xl-turbo` | DiT | XL (4B params), turbo |
| `acestep-5Hz-lm-0.6B` | LM | Smaller language model (faster) |
| `acestep-5Hz-lm-1.7B` | LM | Larger language model (better quality) |

In [ ]:
#@title ## 4. Initialize Models { display-mode: "form" }

#@markdown ### DiT Model Selection
DIT_MODEL = "acestep-v15-turbo" #@param ["acestep-v15-turbo", "acestep-v15-base", "acestep-v15-sft", "acestep-v15-xl-turbo"]

#@markdown ### LM Model Selection
LM_MODEL = "acestep-5Hz-lm-0.6B" #@param ["acestep-5Hz-lm-0.6B", "acestep-5Hz-lm-1.7B"]

#@markdown ### Performance Options
ENABLE_CPU_OFFLOAD = True #@param {type:"boolean"}
OFFLOAD_DIT_TO_CPU = False #@param {type:"boolean"}
USE_FLASH_ATTENTION = False #@param {type:"boolean"}

import os
import sys
import time

# Ensure project is on the path
PROJECT_DIR = "/content/acestep-v1.5"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler

print("=" * 60)
print("Initializing ACE-Step v1.5 Models")
print("=" * 60)

# Detect GPU memory
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {gpu_mem_gb:.1f} GB")

    # Auto-enable offload for GPUs with less than 16 GB
    if gpu_mem_gb < 16 and not ENABLE_CPU_OFFLOAD:
        print("\u26a0\ufe0f  GPU has <16 GB VRAM — enabling CPU offload automatically")
        ENABLE_CPU_OFFLOAD = True

# ---- Initialize DiT Handler ----
print(f"\n[1/2] Loading DiT model: {DIT_MODEL}")
print("  (First run will download from HuggingFace Hub — this may take 10-20 minutes)")

dit_handler = AceStepHandler(persistent_storage_path=PROJECT_DIR)

start_time = time.time()
init_status, enable_generate = dit_handler.initialize_service(
    project_root=PROJECT_DIR,
    config_path=DIT_MODEL,
    device="auto",
    use_flash_attention=USE_FLASH_ATTENTION,
    compile_model=False,
    offload_to_cpu=ENABLE_CPU_OFFLOAD,
    offload_dit_to_cpu=OFFLOAD_DIT_TO_CPU,
)
dit_load_time = time.time() - start_time

if not enable_generate:
    print(f"\u274c DiT initialization failed:\n{init_status}")
    raise RuntimeError("DiT model initialization failed")

print(f"\u2705 DiT loaded in {dit_load_time:.1f}s")
print(init_status)

# ---- Initialize LM Handler ----
print(f"\n[2/2] Loading 5Hz LM: {LM_MODEL}")
print("  (Tokenizer loading may take 60-90 seconds)")

llm_handler = LLMHandler(persistent_storage_path=PROJECT_DIR)

checkpoint_dir = dit_handler._get_checkpoint_dir()

# Ensure LM model is downloaded
lm_model_path = os.path.join(checkpoint_dir, LM_MODEL)
if not os.path.exists(lm_model_path):
    print(f"  Downloading {LM_MODEL} from HuggingFace Hub...")
    dit_handler._ensure_model_downloaded(LM_MODEL, checkpoint_dir)

start_time = time.time()
lm_status, lm_success = llm_handler.initialize(
    checkpoint_dir=checkpoint_dir,
    lm_model_path=LM_MODEL,
    backend="pt",  # Use PyTorch backend for Colab compatibility
    device="auto",
    offload_to_cpu=ENABLE_CPU_OFFLOAD,
    dtype=dit_handler.dtype,
)
lm_load_time = time.time() - start_time

if not lm_success:
    print(f"\u26a0\ufe0f  LM initialization failed (text2music will still work without CoT):")
    print(f"  {lm_status}")
else:
    print(f"\u2705 LM loaded in {lm_load_time:.1f}s")
    print(lm_status)

# ---- Create output directory ----
OUTPUT_DIR = "/content/acestep_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "=" * 60)
print("\u2705 Model initialization complete!")
print(f"\u2705 Output directory: {OUTPUT_DIR}")
print("=" * 60)

## Cell 5: Text-to-Music Generation

Generate music from a text prompt and optional lyrics. This is the core **Text2Music** feature.

### Key Parameters
- **caption**: Text description of the desired music style/mood
- **lyrics**: Lyrics for vocal tracks (use `[Instrumental]` for no vocals)
- **bpm**: Beats per minute (30–300, or `None` for auto)
- **keyscale**: Musical key (e.g., `"C Major"`, `"Am"`)
- **duration**: Target length in seconds (10–600, or `-1` for auto)
- **inference_steps**: Diffusion steps (8 for turbo, 32–100 for base)
- **seed**: `-1` for random, or an integer for reproducibility
- **thinking**: Enable LM Chain-of-Thought reasoning for better quality

In [ ]:
#@title ## 5. Text-to-Music Generation { display-mode: "form" }

from acestep.inference import generate_music, GenerationParams, GenerationConfig
import IPython.display as ipd
import time

#@markdown ### Music Description
CAPTION = "A dreamy, atmospheric synthwave track with lush pads, arpeggiated synthesizers, and a slow-building crescendo. Ethereal and cinematic feel, perfect for a sci-fi movie soundtrack." #@param {type:"string"}
LYRICS = "[Instrumental]" #@param {type:"string"}

#@markdown ### Music Metadata
BPM = 90 #@param {type:"slider", min:30, max:300, step:1}
KEYSCALE = "C Major" #@param {type:"string"}
DURATION = 30 #@param {type:"slider", min:10, max:120, step:5}
VOCAL_LANGUAGE = "unknown" #@param ["unknown", "en", "zh", "ja", "ko", "fr", "de", "es", "it", "pt", "ru"]

#@markdown ### Generation Settings
INFERENCE_STEPS = 8 #@param {type:"slider", min:4, max:100, step:1}
SEED = -1 #@param {type:"integer"}
THINKING = True #@param {type:"boolean"}
BATCH_SIZE = 1 #@param {type:"slider", min:1, max:4, step:1}

print("\u266b Generating music with Text2Music mode...\n")

# Set up generation parameters
params = GenerationParams(
    task_type="text2music",
    caption=CAPTION,
    lyrics=LYRICS,
    bpm=BPM if BPM > 0 else None,
    keyscale=KEYSCALE,
    duration=DURATION if DURATION > 0 else -1.0,
    vocal_language=VOCAL_LANGUAGE,
    inference_steps=INFERENCE_STEPS,
    seed=SEED,
    thinking=THINKING,
    use_cot_metas=True,
    use_cot_caption=True,
    use_cot_language=True,
)

config = GenerationConfig(
    batch_size=BATCH_SIZE,
    use_random_seed=(SEED == -1),
    audio_format="mp3",
)

# Generate
start_time = time.time()
result = generate_music(
    dit_handler=dit_handler,
    llm_handler=llm_handler,
    params=params,
    config=config,
    save_dir=OUTPUT_DIR,
)
gen_time = time.time() - start_time

# Display results
if result.success:
    print(f"\n\u2705 Generation complete in {gen_time:.1f}s")
    print(f"Status: {result.status_message}")

    for i, audio_info in enumerate(result.audios):
        audio_path = audio_info.get("path", "")
        sample_rate = audio_info.get("sample_rate", 48000)
        print(f"\n\u266b Audio {i+1}:")
        print(f"  Path: {audio_path}")
        print(f"  Sample rate: {sample_rate} Hz")
        if audio_path and os.path.exists(audio_path):
            ipd.display(ipd.Audio(audio_path))
        else:
            # Try playing from tensor
            tensor = audio_info.get("tensor")
            if tensor is not None:
                ipd.display(ipd.Audio(tensor.numpy(), rate=sample_rate))
else:
    print(f"\u274c Generation failed: {result.error}")
    print(result.status_message)

## Cell 6: Cover Generation

Create a cover version of an existing audio file with modified style, BPM, or key. Upload your source audio file below.

### How Cover Mode Works
1. The source audio is encoded into latent representations
2. The DiT model generates new audio conditioned on both the source audio and your text prompt
3. `audio_cover_strength` controls how closely the cover follows the source (1.0 = faithful, 0.2 = mostly new style)

In [ ]:
#@title ## 6. Cover Generation { display-mode: "form" }

from google.colab import files
import shutil

#@markdown ### Upload Source Audio
UPLOAD_AUDIO = True #@param {type:"boolean"}

#@markdown ### Cover Style
COVER_CAPTION = "A jazz trio cover with smooth piano, upright bass, and brushed drums" #@param {type:"string"}
COVER_LYRICS = "[Instrumental]" #@param {type:"string"}
COVER_BPM = None #@param {type:"raw"}
COVER_KEYSCALE = "" #@param {type:"string"}

#@markdown ### Cover Strength (0.0 = ignore source, 1.0 = faithful copy)
AUDIO_COVER_STRENGTH = 0.7 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ### Generation Settings
COVER_STEPS = 8 #@param {type:"slider", min:4, max:100, step:1}
COVER_SEED = -1 #@param {type:"integer"}
COVER_DURATION = -1 #@param {type:"slider", min:-1, max:120, step:5}

# Upload or specify source audio
source_audio_path = None

if UPLOAD_AUDIO:
    print("Please upload a source audio file (mp3, wav, flac, etc.)...")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        source_audio_path = os.path.join(OUTPUT_DIR, f"source_{uploaded_name}")
        shutil.move(uploaded_name, source_audio_path)
        print(f"\u2705 Source audio saved to: {source_audio_path}")
    else:
        print("\u274c No file uploaded")

if source_audio_path and os.path.exists(source_audio_path):
    print("\n\u266b Generating cover version...\n")

    params = GenerationParams(
        task_type="cover",
        caption=COVER_CAPTION,
        lyrics=COVER_LYRICS,
        bpm=COVER_BPM if isinstance(COVER_BPM, int) else None,
        keyscale=COVER_KEYSCALE,
        duration=COVER_DURATION if COVER_DURATION > 0 else -1.0,
        inference_steps=COVER_STEPS,
        seed=COVER_SEED,
        src_audio=source_audio_path,
        audio_cover_strength=AUDIO_COVER_STRENGTH,
        thinking=False,  # LM is skipped for cover tasks
    )

    config = GenerationConfig(
        batch_size=1,
        use_random_seed=(COVER_SEED == -1),
        audio_format="mp3",
    )

    start_time = time.time()
    result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=params,
        config=config,
        save_dir=OUTPUT_DIR,
    )
    gen_time = time.time() - start_time

    if result.success:
        print(f"\n\u2705 Cover generated in {gen_time:.1f}s")
        for i, audio_info in enumerate(result.audios):
            audio_path = audio_info.get("path", "")
            sr = audio_info.get("sample_rate", 48000)
            if audio_path and os.path.exists(audio_path):
                ipd.display(ipd.Audio(audio_path))
            else:
                tensor = audio_info.get("tensor")
                if tensor is not None:
                    ipd.display(ipd.Audio(tensor.numpy(), rate=sr))
    else:
        print(f"\u274c Cover generation failed: {result.error}")
else:
    print("\u26a0\ufe0f  No source audio available. Upload a file to use cover mode.")

## Cell 7: Repaint — Edit Specific Sections

Repaint mode allows you to re-generate a specific time region of an existing audio file while preserving the rest. This is useful for:
- Fixing a section that doesn't sound right
- Changing the bridge or chorus
- Extending a song by repainting the ending

### Parameters
- `repainting_start`: Start time in seconds of the region to repaint
- `repainting_end`: End time in seconds (-1 = until the end of the audio)

In [ ]:
#@title ## 7. Repaint — Edit Specific Sections { display-mode: "form" }

from google.colab import files
import shutil

#@markdown ### Upload Source Audio
REPAINT_UPLOAD = True #@param {type:"boolean"}

#@markdown ### Repaint Region (seconds)
REPAINT_START = 10.0 #@param {type:"slider", min:0, max:120, step:0.5}
REPAINT_END = 20.0 #@param {type:"slider", min:0, max:120, step:0.5}

#@markdown ### New Content Description
REPAINT_CAPTION = "An energetic drum fill leading into a powerful guitar solo" #@param {type:"string"}
REPAINT_LYRICS = "" #@param {type:"string"}

#@markdown ### Generation Settings
REPAINT_STEPS = 8 #@param {type:"slider", min:4, max:100, step:1}
REPAINT_SEED = -1 #@param {type:"integer"}

# Upload source audio
repaint_source_path = None

if REPAINT_UPLOAD:
    print("Upload the audio file you want to repaint...")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        repaint_source_path = os.path.join(OUTPUT_DIR, f"repaint_source_{uploaded_name}")
        shutil.move(uploaded_name, repaint_source_path)
        print(f"\u2705 Source audio saved to: {repaint_source_path}")

if repaint_source_path and os.path.exists(repaint_source_path):
    print(f"\n\u266b Repainting region {REPAINT_START}s - {REPAINT_END}s...\n")

    params = GenerationParams(
        task_type="repaint",
        caption=REPAINT_CAPTION,
        lyrics=REPAINT_LYRICS,
        inference_steps=REPAINT_STEPS,
        seed=REPAINT_SEED,
        src_audio=repaint_source_path,
        repainting_start=REPAINT_START,
        repainting_end=REPAINT_END if REPAINT_END > REPAINT_START else -1,
        thinking=False,  # LM is skipped for repaint tasks
    )

    config = GenerationConfig(
        batch_size=1,
        use_random_seed=(REPAINT_SEED == -1),
        audio_format="mp3",
    )

    start_time = time.time()
    result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=params,
        config=config,
        save_dir=OUTPUT_DIR,
    )
    gen_time = time.time() - start_time

    if result.success:
        print(f"\n\u2705 Repaint complete in {gen_time:.1f}s")
        for i, audio_info in enumerate(result.audios):
            audio_path = audio_info.get("path", "")
            sr = audio_info.get("sample_rate", 48000)
            if audio_path and os.path.exists(audio_path):
                ipd.display(ipd.Audio(audio_path))
            else:
                tensor = audio_info.get("tensor")
                if tensor is not None:
                    ipd.display(ipd.Audio(tensor.numpy(), rate=sr))
    else:
        print(f"\u274c Repaint failed: {result.error}")
else:
    print("\u26a0\ufe0f  No source audio available. Upload a file to use repaint mode.")

## Cell 8: Remix Song — Cover + Repaint with Style Transfer

This advanced example demonstrates how to **remix** a song by combining cover mode with repaint mode for style transfer. The workflow is:

1. **Load** a source song
2. **Cover** with reduced `audio_cover_strength` to transfer the style while keeping the structure
3. **Repaint** specific sections to further modify the result

### Key Concept: `audio_cover_strength`
| Value | Effect |
|-------|--------|
| `1.0` | Faithful cover — preserves almost everything from source |
| `0.5` | Balanced remix — keeps melody, changes style |
| `0.2` | Style hint — mostly new generation with source flavor |
| `0.0` | Ignores source audio completely |

In [ ]:
#@title ## 8. Remix Song — Cover + Repaint Style Transfer { display-mode: "form" }

from google.colab import files
import shutil

#@markdown ### Step 1: Upload Source Song
REMIX_UPLOAD = True #@param {type:"boolean"}

#@markdown ### Step 2: Cover Settings (Style Transfer)
REMIX_STYLE_CAPTION = "A lo-fi hip hop chill beat with vinyl crackle, mellow Rhodes piano, and a laid-back boom-bap drum pattern" #@param {type:"string"}
REMIX_STYLE_LYRICS = "[Instrumental]" #@param {type:"string"}
REMIX_COVER_STRENGTH = 0.3 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
REMIX_BPM = 85 #@param {type:"slider", min:30, max:300, step:1}
REMIX_KEYSCALE = "Eb minor" #@param {type:"string"}
REMIX_DURATION = 30 #@param {type:"slider", min:10, max:120, step:5}

#@markdown ### Step 3: Repaint Settings (Optional Section Edit)
ENABLE_REPAINT = False #@param {type:"boolean"}
REMIX_REPAINT_START = 15.0 #@param {type:"slider", min:0, max:120, step:0.5}
REMIX_REPAINT_END = 25.0 #@param {type:"slider", min:0, max:120, step:0.5}
REMIX_REPAINT_CAPTION = "A smooth key change with rising strings and a filter sweep" #@param {type:"string"}

#@markdown ### Generation Settings
REMIX_STEPS = 8 #@param {type:"slider", min:4, max:100, step:1}
REMIX_SEED = -1 #@param {type:"integer"}

# ---- Upload source song ----
remix_source_path = None

if REMIX_UPLOAD:
    print("Upload the song you want to remix...")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        remix_source_path = os.path.join(OUTPUT_DIR, f"remix_source_{uploaded_name}")
        shutil.move(uploaded_name, remix_source_path)
        print(f"\u2705 Source audio saved to: {remix_source_path}")

if not (remix_source_path and os.path.exists(remix_source_path)):
    print("\u26a0\ufe0f  No source audio. Using a generated text2music sample as the source for demo.")
    print("  Generating a 15-second source track first...\n")

    # Generate a source track to remix
    source_params = GenerationParams(
        task_type="text2music",
        caption="An upbeat electronic dance track with pulsing bass, synth leads, and a four-on-the-floor beat",
        lyrics="[Instrumental]",
        bpm=128,
        keyscale="A minor",
        duration=15,
        inference_steps=8,
        seed=42,
        thinking=True,
    )
    source_config = GenerationConfig(batch_size=1, use_random_seed=False, audio_format="mp3")

    source_result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=source_params,
        config=source_config,
        save_dir=OUTPUT_DIR,
    )

    if source_result.success and source_result.audios:
        # Use the generated audio's path as the remix source
        remix_source_path = source_result.audios[0].get("path", "")
        print(f"\u2705 Source track generated: {remix_source_path}")
        print("\nOriginal source track:")
        if remix_source_path and os.path.exists(remix_source_path):
            ipd.display(ipd.Audio(remix_source_path))
    else:
        print(f"\u274c Could not generate source track: {source_result.error}")
        remix_source_path = None

# ---- Step 2: Cover with style transfer ----
if remix_source_path and os.path.exists(remix_source_path):
    print("\n" + "=" * 60)
    print("STEP 2: Style Transfer (Cover Mode)")
    print("=" * 60)
    print(f"Style: {REMIX_STYLE_CAPTION}")
    print(f"Cover strength: {REMIX_COVER_STRENGTH}")
    print(f"BPM: {REMIX_BPM}, Key: {REMIX_KEYSCALE}")
    print()

    cover_params = GenerationParams(
        task_type="cover",
        caption=REMIX_STYLE_CAPTION,
        lyrics=REMIX_STYLE_LYRICS,
        bpm=REMIX_BPM,
        keyscale=REMIX_KEYSCALE,
        duration=REMIX_DURATION if REMIX_DURATION > 0 else -1.0,
        inference_steps=REMIX_STEPS,
        seed=REMIX_SEED,
        src_audio=remix_source_path,
        audio_cover_strength=REMIX_COVER_STRENGTH,
        thinking=False,
    )

    cover_config = GenerationConfig(
        batch_size=1,
        use_random_seed=(REMIX_SEED == -1),
        audio_format="mp3",
    )

    start_time = time.time()
    cover_result = generate_music(
        dit_handler=dit_handler,
        llm_handler=llm_handler,
        params=cover_params,
        config=cover_config,
        save_dir=OUTPUT_DIR,
    )
    cover_time = time.time() - start_time

    if cover_result.success and cover_result.audios:
        cover_audio_path = cover_result.audios[0].get("path", "")
        print(f"\u2705 Cover/remix generated in {cover_time:.1f}s")
        print("\nRemixed audio (Cover with style transfer):")
        if cover_audio_path and os.path.exists(cover_audio_path):
            ipd.display(ipd.Audio(cover_audio_path))
        else:
            tensor = cover_result.audios[0].get("tensor")
            if tensor is not None:
                ipd.display(ipd.Audio(tensor.numpy(), rate=48000))
    else:
        print(f"\u274c Cover generation failed: {cover_result.error if cover_result else 'Unknown error'}")
        cover_audio_path = None

    # ---- Step 3: Optional Repaint ----
    if ENABLE_REPAINT and cover_audio_path and os.path.exists(cover_audio_path):
        print("\n" + "=" * 60)
        print("STEP 3: Repaint Section")
        print("=" * 60)
        print(f"Region: {REMIX_REPAINT_START}s - {REMIX_REPAINT_END}s")
        print(f"Description: {REMIX_REPAINT_CAPTION}")
        print()

        repaint_params = GenerationParams(
            task_type="repaint",
            caption=REMIX_REPAINT_CAPTION,
            lyrics="",
            inference_steps=REMIX_STEPS,
            seed=REMIX_SEED,
            src_audio=cover_audio_path,
            repainting_start=REMIX_REPAINT_START,
            repainting_end=REMIX_REPAINT_END if REMIX_REPAINT_END > REMIX_REPAINT_START else -1,
            thinking=False,
        )

        repaint_config = GenerationConfig(
            batch_size=1,
            use_random_seed=(REMIX_SEED == -1),
            audio_format="mp3",
        )

        start_time = time.time()
        repaint_result = generate_music(
            dit_handler=dit_handler,
            llm_handler=llm_handler,
            params=repaint_params,
            config=repaint_config,
            save_dir=OUTPUT_DIR,
        )
        repaint_time = time.time() - start_time

        if repaint_result.success and repaint_result.audios:
            final_path = repaint_result.audios[0].get("path", "")
            print(f"\u2705 Final remix with repaint in {repaint_time:.1f}s")
            print("\nFinal remixed + repainted audio:")
            if final_path and os.path.exists(final_path):
                ipd.display(ipd.Audio(final_path))
            else:
                tensor = repaint_result.audios[0].get("tensor")
                if tensor is not None:
                    ipd.display(ipd.Audio(tensor.numpy(), rate=48000))
        else:
            print(f"\u274c Repaint failed: {repaint_result.error}")
    else:
        if not ENABLE_REPAINT:
            print("\nSkipping repaint step (ENABLE_REPAINT is False)")
            print("The cover-style transfer result above is your final remix!")
        print(f"\n\u2705 Remix complete!")

## Cell 9: Launch Gradio Web UI

Launch the full Gradio interface for interactive music generation. This provides a user-friendly web UI with all features accessible through clickable controls.

**Note**: The Gradio UI will create a public share link that you can open in a new browser tab.

In [ ]:
#@title ## 9. Launch Gradio Web UI { display-mode: "form" }

SHARE_PUBLIC = True #@param {type:"boolean"}
SERVER_PORT = 7860 #@param {type:"integer"}

import sys
import os

PROJECT_DIR = "/content/acestep-v1.5"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Disable Gradio analytics
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
# Clear proxy settings
for proxy_var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY']:
    os.environ.pop(proxy_var, None)
# Disable torchcodec
os.environ["TORCHAUDIO_USE_TORCHCODEC"] = "0"

from acestep.dataset_handler import DatasetHandler
from acestep.gradio_ui import create_gradio_interface

print("Creating Gradio interface...")

# Prepare initialization parameters for pre-initialized models
init_params = {
    'pre_initialized': True,
    'service_mode': True,
    'checkpoint': None,
    'config_path': DIT_MODEL,
    'config_path_2': None,
    'device': 'auto',
    'init_llm': llm_handler.llm_initialized,
    'lm_model_path': LM_MODEL,
    'backend': 'pt',
    'use_flash_attention': USE_FLASH_ATTENTION,
    'offload_to_cpu': ENABLE_CPU_OFFLOAD,
    'offload_dit_to_cpu': OFFLOAD_DIT_TO_CPU,
    'init_status': 'Colab initialization',
    'enable_generate': enable_generate,
    'dit_handler': dit_handler,
    'dit_handler_2': None,
    'available_dit_models': [DIT_MODEL],
    'llm_handler': llm_handler,
    'language': 'en',
    'persistent_storage_path': PROJECT_DIR,
    'debug_ui': False,
}

dataset_handler = DatasetHandler()

demo = create_gradio_interface(
    dit_handler,
    llm_handler,
    dataset_handler,
    init_params=init_params,
    language='en'
)

demo.queue(max_size=5)

print(f"\n\u2705 Launching Gradio on port {SERVER_PORT}...")
if SHARE_PUBLIC:
    print("A public share link will be generated — open it in a new tab!")

demo.launch(
    server_name="0.0.0.0",
    server_port=SERVER_PORT,
    share=SHARE_PUBLIC,
    show_error=True,
)

## Cell 10: Download Generated Audio Files

Download all generated audio files to your local machine. Files are saved in the output directory from previous generation cells.

In [ ]:
#@title ## 10. Download Generated Audio Files { display-mode: "form" }

from google.colab import files
import glob

#@markdown ### Download Options
DOWNLOAD_ALL = True #@param {type:"boolean"}
SPECIFIC_FILE = "" #@param {type:"string"}
AS_ZIP = True #@param {type:"boolean"}

OUTPUT_DIR = "/content/acestep_output"

# Find all audio files
audio_extensions = ['*.mp3', '*.wav', '*.flac']
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(OUTPUT_DIR, ext)))

if not audio_files:
    print("\u26a0\ufe0f  No audio files found in output directory.")
    print("  Run a generation cell first!")
else:
    print(f"Found {len(audio_files)} audio file(s):")
    for f in audio_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

    if SPECIFIC_FILE:
        # Download specific file
        target = os.path.join(OUTPUT_DIR, SPECIFIC_FILE)
        if os.path.exists(target):
            print(f"\nDownloading: {SPECIFIC_FILE}")
            files.download(target)
        else:
            print(f"\u274c File not found: {target}")
    elif AS_ZIP and len(audio_files) > 1:
        # Download as ZIP archive
        import zipfile
        zip_path = os.path.join(OUTPUT_DIR, "acestep_generated.zip")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in audio_files:
                zf.write(f, os.path.basename(f))
        zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
        print(f"\nDownloading ZIP archive ({zip_size_mb:.1f} MB)...")
        files.download(zip_path)
    elif DOWNLOAD_ALL:
        # Download files individually
        print("\nDownloading files...")
        for f in audio_files:
            print(f"  Downloading: {os.path.basename(f)}")
            files.download(f)
    else:
        print("\nSet DOWNLOAD_ALL=True or specify a SPECIFIC_FILE to download.")

print("\n\u2705 Done!")

---

## 📋 Tips & Troubleshooting

### Memory Management
- If you get **CUDA Out of Memory** errors, try:
  - Enabling `OFFLOAD_DIT_TO_CPU = True` in Cell 4
  - Reducing `BATCH_SIZE` to 1
  - Reducing `DURATION` (shorter clips use less memory)
  - Using `acestep-v15-turbo` instead of XL models

### Generation Quality
- **Turbo models** (8 steps) are fast but may have less detail
- **Base model** (32–100 steps) produces higher quality but is much slower
- Enable `THINKING = True` for the LM to generate better audio codes via Chain-of-Thought
- Use `seed` parameter for reproducible results

### Cover & Remix Tips
- `audio_cover_strength = 0.2–0.3`: Good for dramatic style changes
- `audio_cover_strength = 0.5–0.7`: Balanced remix
- `audio_cover_strength = 0.8–1.0`: Faithful cover with minor changes

### Supported Languages
Vocals can be generated in: `ar`, `az`, `bg`, `bn`, `ca`, `cs`, `da`, `de`, `el`, `en`, `es`, `fa`, `fi`, `fr`, `he`, `hi`, `hr`, `ht`, `hu`, `id`, `is`, `it`, `ja`, `ko`, `la`, `lt`, `ms`, `ne`, `nl`, `no`, `pa`, `pl`, `pt`, `ro`, `ru`, `sa`, `sk`, `sr`, `sv`, `sw`, `ta`, `te`, `th`, `tl`, `tr`, `uk`, `ur`, `vi`, `yue`, `zh`

### More Info
- [GitHub Repository](https://github.com/BF667/acestep-v1.5)
- [HuggingFace Model](https://huggingface.co/ACE-Step/Ace-Step1.5)
- [API Documentation](https://github.com/BF667/acestep-v1.5/blob/main/docs/en/API.md)
